In [ ]:
"""
ResNet50 + 自适应多尺度注意力融合 语义分割网络
基于用户提供的原始架构改进
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import torch.utils.model_zoo as model_zoo
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import os
import random
from model import CBAM


# ==================== 1. 基础组件：Bottleneck ====================
class Bottleneck(nn.Module):
    """ResNet50的瓶颈块 (引用自原文件)"""
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        return self.relu(out)

# ==================== 2. 核心改进：自适应双向引导交互模块 ====================
class AdaptiveBidirectionalInteraction(nn.Module):
    """
    自适应多尺度注意力融合机制：
    - 路径1: 浅层空间 As -> Gsd 引导深层通道
    - 路径2: 深层语义 Ac -> Gds 优化浅层空间
    """
    def __init__(self, s_channels, d_channels, reduction=16):
        super(AdaptiveBidirectionalInteraction, self).__init__()

        # --- 浅层分支 (Spatial Focus) ---
        self.s_conv7x7 = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

        # As -> Gsd (Spatial to Deep Channel Guidance)
        self.s_to_d_mlp = nn.Sequential(
            nn.Linear(1, d_channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(d_channels // reduction, d_channels)
        )

        # --- 深层分支 (Channel Semantic) ---
        self.d_shared_mlp = nn.Sequential(
            nn.Linear(d_channels, d_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(d_channels // reduction, d_channels, bias=False)
        )

        # Ac -> Gds (Semantic to Shallow Spatial Guidance)
        self.d_to_s_mlp = nn.Sequential(
            nn.Linear(d_channels, d_channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(d_channels // reduction, 1),
            nn.Sigmoid()
        )

        mid_channels = 256
        self.d_to_s_conv = nn.Sequential(
            nn.Conv2d(d_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, 1, kernel_size=1, bias=False),
            nn.Sigmoid()
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, Fs, Fd):
        # 1. 深层通道注意力计算 (Ac)
        avg_p = F.adaptive_avg_pool2d(Fd, 1).view(Fd.size(0), -1)
        max_p = F.adaptive_max_pool2d(Fd, 1).view(Fd.size(0), -1)
        d_stats = self.d_shared_mlp(avg_p) + self.d_shared_mlp(max_p)
        Ac = self.sigmoid(d_stats)
        # 浅层空间使用池化
        s_avg = torch.mean(Fs, dim=1, keepdim=True)
        s_max, _ = torch.max(Fs, dim=1, keepdim=True)
        s_desc = torch.cat([s_avg, s_max], dim=1) # 此时为B,2,H,W

        # 2. 深层对浅层的语义引导 (Ac -> Gds)

        # Gds = self.d_to_s_mlp(Ac)
        Gds = self.d_to_s_conv(Fd * Ac.view(Fd.size(0), Fd.size(1), 1, 1))

        # 3. 浅层空间注意力计算 (As), 注入 Gds
        Gds_up = F.interpolate(Gds, size=(Fs.size(2), Fs.size(3)), mode='bilinear', align_corners=False)

        #这种是使用了
        # s_desc_fused = s_desc * Gds.view(-1, 1, 1, 1) # 语义引导注入
        s_desc_fused = s_desc *Gds_up # 语义引导注入
        As = self.sigmoid(self.s_conv7x7(s_desc_fused))
        Fs_out = Fs * As

        # 4. 浅层对深层的空间引导 (As -> Gsd)
        s_guide_vec = F.adaptive_avg_pool2d(As, 1).view(As.size(0), -1)
        Gsd = self.s_to_d_mlp(s_guide_vec)

        # 5. 深层最终权重融合
        final_Ac = self.sigmoid(d_stats + Gsd).view(Fd.size(0), Fd.size(1), 1, 1)
        Fd_out = Fd * final_Ac

        return Fs_out, Fd_out


class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        assert kernel_size in (3, 7), 'kernel size must be 3 or 7'
        padding = 3 if kernel_size == 7 else 1
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)


class CBAM(nn.Module):
    def __init__(self, gate_channels, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ChannelGate = ChannelAttention(gate_channels, ratio)
        self.SpatialGate = SpatialAttention(kernel_size)

    def forward(self, x):
        x_out = x * self.ChannelGate(x)
        x_out = x_out * self.SpatialGate(x_out)
        return x_out

# ==================== 4. 数据处理与损失函数 (原样引用自原文件) ====================
class CityscapesDataset(Dataset):
    """Cityscapes数据集加载器"""
    CLASSES = ['road', 'sidewalk', 'building', 'wall', 'fence', 'pole', 'traffic light',
               'traffic sign', 'vegetation', 'terrain', 'sky', 'person', 'rider', 'car',
               'truck', 'bus', 'train', 'motorcycle', 'bicycle']

    CLASS_COLORS = {0: [128, 64, 128], 1: [244, 35, 232], 2: [70, 70, 70], 3: [102, 102, 156],
                    4: [190, 153, 153], 5: [153, 153, 153], 6: [250, 170, 30], 7: [220, 220, 0],
                    8: [107, 142, 35], 9: [152, 251, 152], 10: [70, 130, 180], 11: [220, 20, 60],
                    12: [255, 0, 0], 13: [0, 0, 142], 14: [0, 0, 70], 15: [0, 60, 100],
                    16: [0, 80, 100], 17: [0, 0, 230], 18: [119, 11, 32], 255: [0, 0, 0]}

    LABEL_ID_MAPPING = {7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5, 19: 6, 20: 7, 21: 8, 22: 9,
                        23: 10, 24: 11, 25: 12, 26: 13, 27: 14, 28: 15, 31: 16, 32: 17, 33: 18}

    def __init__(self, root, split='train', transform=None, target_size=(1024, 512)):
        self.root, self.split, self.transform, self.target_size = root, split, transform, target_size
        self.images = self._get_image_paths()
        self.targets = self._get_target_paths()

    def _get_image_paths(self):
        image_dir = os.path.join(self.root, 'leftImg8bit', self.split)
        images = []
        for city in os.listdir(image_dir):
            city_dir = os.path.join(image_dir, city)
            for img_name in os.listdir(city_dir):
                if img_name.endswith('_leftImg8bit.png'): images.append(os.path.join(city_dir, img_name))
        return sorted(images)

    def _get_target_paths(self):
        target_dir = os.path.join(self.root, 'gtFine', self.split)
        targets = []
        for img_path in self.images:
            img_name = os.path.basename(img_path)
            base_name = img_name.replace('_leftImg8bit.png', '')
            city = os.path.basename(os.path.dirname(img_path))
            target_name = f'{base_name}_gtFine_labelIds.png'
            targets.append(os.path.join(target_dir, city, target_name))
        return sorted(targets)

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert('RGB')
        target = Image.open(self.targets[idx])
        orig_size = image.size[::-1]
        image = image.resize(self.target_size, Image.BILINEAR)
        target = target.resize(self.target_size, Image.NEAREST)
        if self.transform: image, target = self.transform(image, target)
        image = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255.0
        target = self._remap_labels(np.array(target))
        return image, target, orig_size, self.images[idx]

    def _remap_labels(self, target):
        remapped = np.full_like(target, 255)
        for old, new in self.LABEL_ID_MAPPING.items(): remapped[target == old] = new
        return torch.from_numpy(remapped).long()

class SimpleTransform:
    def __init__(self, crop_size=(512, 512), flip_prob=0.5):
        self.crop_size, self.flip_prob = crop_size, flip_prob
    def __call__(self, image, target):
        if random.random() < self.flip_prob:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            target = target.transpose(Image.FLIP_LEFT_RIGHT)
        w, h = image.size
        cw, ch = self.crop_size
        left = random.randint(0, w - cw)
        top = random.randint(0, h - ch)
        image = image.crop((left, top, left + cw, top + ch))
        target = target.crop((left, top, left + cw, top + ch))
        return image, target
# ==================== 3. 改进版语义分割网络 ====================

class ResNet50_AdaptiveInteraction_Segmentation(nn.Module):
    def __init__(self, block=Bottleneck, layers=[3, 4, 6, 3], num_classes=19, pretrained=False):
        super(ResNet50_AdaptiveInteraction_Segmentation, self).__init__()
        self.inplanes = 64

        # Encoder 前缀
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet 层级
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.cbam1 = CBAM(256)
        self.cbam2 = CBAM(512)
        self.cbam3 = CBAM(1024)
        self.cbam4 = CBAM(2048)

        # 核心交互模块：连接 Layer 2 (浅层 512ch) 和 Layer 4 (深层 2048ch)
        # self.interaction = AdaptiveBidirectionalInteraction(512, 2048)
        # 针对 Layer 2 and 4
        self.interaction_heavy = AdaptiveBidirectionalInteraction(512, 2048)
        # 针对 Layer 1 and 3
        self.interaction_light = AdaptiveBidirectionalInteraction(256, 1024)

        # 解码器部分 (引用自原文件逻辑)
        self.decoder4 = self._make_decoder_block(2048 + 1024, 256)
        self.decoder3 = self._make_decoder_block(256 + 512, 128)
        self.decoder2 = self._make_decoder_block(128 + 256, 64)
        self.decoder1 = self._make_decoder_block(64 + 64, 32)

        self.final_conv = nn.Conv2d(32, num_classes, kernel_size=1)
        self._initialize_weights()
        if pretrained: self._load_pretrained_weights()

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )
        layers = [block(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks): layers.append(block(self.inplanes, planes))
        return nn.Sequential(*layers)

    def _make_decoder_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True)
        )

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1); m.bias.data.zero_()

    def _load_pretrained_weights(self):
        try:
            pre_dict = model_zoo.load_url('https://download.pytorch.org/models/resnet50-19c8e357.pth')
            model_dict = self.state_dict()
            pre_dict = {k: v for k, v in pre_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
            model_dict.update(pre_dict)
            self.load_state_dict(model_dict)
            print("成功加载预训练权重")
        except: print("预训练加载失败")

    def forward(self, x, size=None):
        # Encoder
        x0 = self.relu(self.bn1(self.conv1(x)))
        x_low = self.maxpool(x0)

        x1 = self.layer1(x_low)
        x1 = self.cbam1(x1)
        x2 = self.layer2(x1)  # 浅层 Fs
        x2 = self.cbam2(x2)
        x3 = self.layer3(x2)
        x3 = self.cbam3(x3)
        x4 = self.layer4(x3)  # 深层 Fd
        x4 = self.cbam4(x4)

        # 核心双向交互
        # x2_enhanced, x4_enhanced = self.interaction(x2, x4)
        # x1_enhanced, x3_enhanced = self.interaction(x1, x3)
        x2_enhanced, x4_enhanced = self.interaction_heavy(x2, x4)
        x1_enhanced, x3_enhanced = self.interaction_light(x1, x3)
        x1_fused = self.relu(x1 + x1_enhanced)
        x2_fused = self.relu(x2 + x2_enhanced)
        x3_fused = self.relu(x3 + x3_enhanced)
        x4_fused = self.relu(x4 + x4_enhanced)


        # Decoder (级联融合)
        h, w = x3.shape[2], x3.shape[3]
        d4 = F.interpolate(x4_fused, size=(h, w), mode='bilinear', align_corners=False)
        d4 = self.decoder4(torch.cat([d4, x3_fused], dim=1))

        h, w = x2_enhanced.shape[2], x2_enhanced.shape[3]
        d3 = F.interpolate(d4, size=(h, w), mode='bilinear', align_corners=False)
        d3 = self.decoder3(torch.cat([d3, x2_fused], dim=1))

        h, w = x1.shape[2], x1.shape[3]
        d2 = F.interpolate(d3, size=(h, w), mode='bilinear', align_corners=False)
        d2 = self.decoder2(torch.cat([d2, x1_enhanced], dim=1))

        h, w = x0.shape[2], x0.shape[3]
        d1 = F.interpolate(d2, size=(h, w), mode='bilinear', align_corners=False)
        d1 = self.decoder1(torch.cat([d1, x0], dim=1))

        out = self.final_conv(d1)
        target_size = size if size is not None else (x.size(2), x.size(3))
        return F.interpolate(out, size=target_size, mode='bilinear', align_corners=False)

# ==================== 4. 数据处理与损失函数 (原样引用自原文件) ====================


class CombinedLoss(nn.Module):
    """组合损失 (引用原文件)"""
    def __init__(self, num_classes=19, ignore_index=255):
        super(CombinedLoss, self).__init__()
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.num_classes = num_classes

    def forward(self, pred, target):
        ce_loss = self.ce(pred, target)
        # Dice Loss 实现... (略，保持与原文件一致)
        return ce_loss

# ==================== 5. 训练器与主逻辑 (原样引用自原文件) ====================

class SegmentationTrainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, scheduler, device, save_dir='checkpoints'):
        self.model, self.train_loader, self.val_loader = model, train_loader, val_loader
        self.criterion, self.optimizer, self.scheduler, self.device = criterion, optimizer, scheduler, device
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        self.train_losses, self.val_losses, self.val_miou_history = [], [], []
        self.epoch_loss_history = []  # 新增：记录每个 Epoch 的系统记录
        self.batch_loss_history = []
        self.best_miou, self.epoch = 0, 0

    def train_epoch(self):
        self.model.train()
        total_loss = 0
        running_loss = 0.0
        log_interval = 50

        for batch_idx, (images, targets, _, _) in enumerate(self.train_loader):
            images, targets = images.to(self.device), targets.to(self.device)
            outputs = self.model(images)
            loss = self.criterion(outputs, targets)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            loss_val = loss.item()
            total_loss += loss_val
            running_loss += loss_val

            # 每 50 个 Batch 记录一次平均 Loss
            if (batch_idx + 1) % log_interval == 0:
                avg_batch_loss = running_loss / log_interval
                self.batch_loss_history.append({
                    'epoch': self.epoch + 1,
                    'batch': batch_idx + 1,
                    'avg_loss': avg_batch_loss
                })
                print(f"  Epoch {self.epoch+1}, Batch {batch_idx+1}, Group Avg Loss: {avg_batch_loss:.4f}")
                running_loss = 0.0  # 重置当前组的累加器
        return total_loss / len(self.train_loader)

    @torch.no_grad()
    def validate(self):
        self.model.eval()
        total_loss, confusion_matrix = 0, np.zeros((19, 19))
        for images, targets, _, _ in self.val_loader:
            images, targets = images.to(self.device), targets.to(self.device)
            outputs = self.model(images)
            total_loss += self.criterion(outputs, targets).item()
            preds = outputs.argmax(dim=1).cpu().numpy()
            for t, p in zip(targets.cpu().numpy(), preds):
                mask = (t != 255)
                label = 19 * t[mask].astype('int') + p[mask]
                confusion_matrix += np.bincount(label, minlength=19**2).reshape(19, 19)
        iu = np.diag(confusion_matrix) / (confusion_matrix.sum(axis=1) + confusion_matrix.sum(axis=0) - np.diag(confusion_matrix) + 1e-10)
        return total_loss / len(self.val_loader), np.mean(iu)

    def train(self, num_epochs):
        for epoch in range(num_epochs):
            self.epoch = epoch
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            # train_loss = self.train_epoch()
            avg_epoch_loss = self.train_epoch()
            val_loss, val_miou = self.validate()
            # 3. 记录到列表（结构化存储）
            self.epoch_loss_history.append({
                'epoch': epoch + 1,
                'train_loss': avg_epoch_loss,
                'val_loss': val_loss,
                'miou': val_miou
            })

            # 4. 显式输出：每组 Epoch 的平均 Loss 总结
            print(f"--- Epoch {epoch+1} Summary ---")
            print(f"Average Train Loss: {avg_epoch_loss:.6f}")
            print(f"Average Val Loss:   {val_loss:.6f}")
            print(f"mIoU:               {val_miou:.4f}")
            if val_miou > self.best_miou:
                self.best_miou = val_miou
                torch.save(self.model.state_dict(), os.path.join(self.save_dir, 'best_model.pth'))
                print("保存最佳模型!")

# ==================== 6. 运行脚本 ====================

def main():
    # 路径配置
    root_dir = r"/home/ps/gzx/D2D_2.0"
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 实例化改进模型
    model = ResNet50_AdaptiveInteraction_Segmentation(num_classes=19, pretrained=True).to(device)

    train_dataset = CityscapesDataset(root_dir, 'train')
    val_dataset = CityscapesDataset(root_dir, 'val')

    # 打印数据集大小
    print(f"训练集图片数量: {len(train_dataset)}")
    print(f"验证集图片数量: {len(val_dataset)}")

    # 数据加载
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8)

    # 打印每个epoch的batch数量
    print(f"训练集每个epoch的batch数量: {len(train_loader)}")
    print(f"验证集每个epoch的batch数量: {len(val_loader)}")

    # 优化器
    criterion = CombinedLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    # 启动训练
    trainer = SegmentationTrainer(model, train_loader, val_loader, criterion, optimizer, None, device)
    trainer.train(num_epochs=0)

if __name__ == "__main__":
    main()

成功加载预训练权重
训练集图片数量: 2975
验证集图片数量: 500
训练集每个epoch的batch数量: 372
验证集每个epoch的batch数量: 63

Epoch 1/50
  Epoch 1, Batch 50, Group Avg Loss: 1.4985
  Epoch 1, Batch 100, Group Avg Loss: 0.7672
  Epoch 1, Batch 150, Group Avg Loss: 0.6548
  Epoch 1, Batch 200, Group Avg Loss: 0.5829
  Epoch 1, Batch 250, Group Avg Loss: 0.5263
  Epoch 1, Batch 300, Group Avg Loss: 0.4957
  Epoch 1, Batch 350, Group Avg Loss: 0.4441
--- Epoch 1 Summary ---
Average Train Loss: 0.693449
Average Val Loss:   0.431326
mIoU:               0.3836
保存最佳模型!

Epoch 2/50
  Epoch 2, Batch 50, Group Avg Loss: 0.4019
  Epoch 2, Batch 100, Group Avg Loss: 0.3881
  Epoch 2, Batch 150, Group Avg Loss: 0.3728
  Epoch 2, Batch 200, Group Avg Loss: 0.3484
  Epoch 2, Batch 250, Group Avg Loss: 0.3476
  Epoch 2, Batch 300, Group Avg Loss: 0.3285
  Epoch 2, Batch 350, Group Avg Loss: 0.3212
--- Epoch 2 Summary ---
Average Train Loss: 0.356070
Average Val Loss:   0.324768
mIoU:               0.4695
保存最佳模型!

Epoch 3/50
  Epoch 3, Batch 50, Group Avg Loss: 0.2916
  Epoch 3, Batch 100, Group Avg Loss: 0.2695
  Epoch 3, Batch 150, Group Avg Loss: 0.2649
  Epoch 3, Batch 200, Group Avg Loss: 0.2714
  Epoch 3, Batch 250, Group Avg Loss: 0.2576
  Epoch 3, Batch 300, Group Avg Loss: 0.2611
  Epoch 3, Batch 350, Group Avg Loss: 0.2569
--- Epoch 3 Summary ---
Average Train Loss: 0.266356
Average Val Loss:   0.273530
mIoU:               0.5307
保存最佳模型!

Epoch 4/50
  Epoch 4, Batch 50, Group Avg Loss: 0.2220
  Epoch 4, Batch 100, Group Avg Loss: 0.2226
  Epoch 4, Batch 150, Group Avg Loss: 0.2214
  Epoch 4, Batch 200, Group Avg Loss: 0.2187
  Epoch 4, Batch 250, Group Avg Loss: 0.2186
  Epoch 4, Batch 300, Group Avg Loss: 0.2065
  Epoch 4, Batch 350, Group Avg Loss: 0.2183
--- Epoch 4 Summary ---
Average Train Loss: 0.218394
Average Val Loss:   0.289675
mIoU:               0.5518
保存最佳模型!

Epoch 5/50
  Epoch 5, Batch 50, Group Avg Loss: 0.1971
  Epoch 5, Batch 100, Group Avg Loss: 0.1960
  Epoch 5, Batch 150, Group Avg Loss: 0.1914
  Epoch 5, Batch 200, Group Avg Loss: 0.2036
  Epoch 5, Batch 250, Group Avg Loss: 0.2087
  Epoch 5, Batch 300, Group Avg Loss: 0.1875
  Epoch 5, Batch 350, Group Avg Loss: 0.1935
--- Epoch 5 Summary ---
Average Train Loss: 0.196550
Average Val Loss:   0.245142
mIoU:               0.5537
保存最佳模型!

Epoch 6/50
  Epoch 6, Batch 50, Group Avg Loss: 0.1672
  Epoch 6, Batch 100, Group Avg Loss: 0.1804
  Epoch 6, Batch 150, Group Avg Loss: 0.1660
  Epoch 6, Batch 200, Group Avg Loss: 0.1707
  Epoch 6, Batch 250, Group Avg Loss: 0.1611
  Epoch 6, Batch 300, Group Avg Loss: 0.1562
  Epoch 6, Batch 350, Group Avg Loss: 0.1579
--- Epoch 6 Summary ---
Average Train Loss: 0.164838
Average Val Loss:   0.224509
mIoU:               0.6221
保存最佳模型!

Epoch 7/50
  Epoch 7, Batch 50, Group Avg Loss: 0.1423
  Epoch 7, Batch 100, Group Avg Loss: 0.1482
  Epoch 7, Batch 150, Group Avg Loss: 0.1405
  Epoch 7, Batch 200, Group Avg Loss: 0.1477
  Epoch 7, Batch 250, Group Avg Loss: 0.1420
  Epoch 7, Batch 300, Group Avg Loss: 0.1467
  Epoch 7, Batch 350, Group Avg Loss: 0.1645
--- Epoch 7 Summary ---
Average Train Loss: 0.148256
Average Val Loss:   0.251307
mIoU:               0.5496

Epoch 8/50
  Epoch 8, Batch 50, Group Avg Loss: 0.1385
  Epoch 8, Batch 100, Group Avg Loss: 0.1430
  Epoch 8, Batch 150, Group Avg Loss: 0.1398
  Epoch 8, Batch 200, Group Avg Loss: 0.1348
  Epoch 8, Batch 250, Group Avg Loss: 0.1364
  Epoch 8, Batch 300, Group Avg Loss: 0.1393
  Epoch 8, Batch 350, Group Avg Loss: 0.1309
--- Epoch 8 Summary ---
Average Train Loss: 0.137256
Average Val Loss:   0.227333
mIoU:               0.6112

Epoch 9/50
  Epoch 9, Batch 50, Group Avg Loss: 0.1286
  Epoch 9, Batch 100, Group Avg Loss: 0.1290
  Epoch 9, Batch 150, Group Avg Loss: 0.1377
  Epoch 9, Batch 200, Group Avg Loss: 0.1425
  Epoch 9, Batch 250, Group Avg Loss: 0.1337
  Epoch 9, Batch 300, Group Avg Loss: 0.1414
  Epoch 9, Batch 350, Group Avg Loss: 0.1433
--- Epoch 9 Summary ---
Average Train Loss: 0.136883
Average Val Loss:   0.227210
mIoU:               0.6144

Epoch 10/50
  Epoch 10, Batch 50, Group Avg Loss: 0.1237
  Epoch 10, Batch 100, Group Avg Loss: 0.1274
  Epoch 10, Batch 150, Group Avg Loss: 0.1218
  Epoch 10, Batch 200, Group Avg Loss: 0.1226
  Epoch 10, Batch 250, Group Avg Loss: 0.1224
  Epoch 10, Batch 300, Group Avg Loss: 0.1216
  Epoch 10, Batch 350, Group Avg Loss: 0.1201
--- Epoch 10 Summary ---
Average Train Loss: 0.122351
Average Val Loss:   0.209580
mIoU:               0.6417
保存最佳模型!

Epoch 11/50
  Epoch 11, Batch 50, Group Avg Loss: 0.1117
  Epoch 11, Batch 100, Group Avg Loss: 0.1094
  Epoch 11, Batch 150, Group Avg Loss: 0.1064
  Epoch 11, Batch 200, Group Avg Loss: 0.1039
  Epoch 11, Batch 250, Group Avg Loss: 0.1067
  Epoch 11, Batch 300, Group Avg Loss: 0.1026
  Epoch 11, Batch 350, Group Avg Loss: 0.1068
--- Epoch 11 Summary ---
Average Train Loss: 0.106635
Average Val Loss:   0.213890
mIoU:               0.6292

Epoch 12/50
  Epoch 12, Batch 50, Group Avg Loss: 0.0971
  Epoch 12, Batch 100, Group Avg Loss: 0.0950
  Epoch 12, Batch 150, Group Avg Loss: 0.1015
  Epoch 12, Batch 200, Group Avg Loss: 0.0983
  Epoch 12, Batch 250, Group Avg Loss: 0.1005
  Epoch 12, Batch 300, Group Avg Loss: 0.0985
  Epoch 12, Batch 350, Group Avg Loss: 0.0978
--- Epoch 12 Summary ---
Average Train Loss: 0.098717
Average Val Loss:   0.204890
mIoU:               0.6419
保存最佳模型!

Epoch 13/50
  Epoch 13, Batch 50, Group Avg Loss: 0.0975
  Epoch 13, Batch 100, Group Avg Loss: 0.1015
  Epoch 13, Batch 150, Group Avg Loss: 0.1083
  Epoch 13, Batch 200, Group Avg Loss: 0.1068
  Epoch 13, Batch 250, Group Avg Loss: 0.1055
  Epoch 13, Batch 300, Group Avg Loss: 0.1013
  Epoch 13, Batch 350, Group Avg Loss: 0.1008
--- Epoch 13 Summary ---
Average Train Loss: 0.103353
Average Val Loss:   0.212884
mIoU:               0.6393

Epoch 14/50
  Epoch 14, Batch 50, Group Avg Loss: 0.0977
  Epoch 14, Batch 100, Group Avg Loss: 0.0950
  Epoch 14, Batch 150, Group Avg Loss: 0.0995
  Epoch 14, Batch 200, Group Avg Loss: 0.0989
  Epoch 14, Batch 250, Group Avg Loss: 0.1062
  Epoch 14, Batch 300, Group Avg Loss: 0.0988
  Epoch 14, Batch 350, Group Avg Loss: 0.1047
--- Epoch 14 Summary ---
Average Train Loss: 0.100287
Average Val Loss:   0.222897
mIoU:               0.6352

Epoch 15/50
  Epoch 15, Batch 50, Group Avg Loss: 0.1103
  Epoch 15, Batch 100, Group Avg Loss: 0.1228
  Epoch 15, Batch 150, Group Avg Loss: 0.1266
  Epoch 15, Batch 200, Group Avg Loss: 0.1305
  Epoch 15, Batch 250, Group Avg Loss: 0.1128
  Epoch 15, Batch 300, Group Avg Loss: 0.1137
  Epoch 15, Batch 350, Group Avg Loss: 0.1062
--- Epoch 15 Summary ---
Average Train Loss: 0.117140
Average Val Loss:   0.201045
mIoU:               0.6452
保存最佳模型!

Epoch 16/50
  Epoch 16, Batch 50, Group Avg Loss: 0.0927
  Epoch 16, Batch 100, Group Avg Loss: 0.0919
  Epoch 16, Batch 150, Group Avg Loss: 0.0926
  Epoch 16, Batch 200, Group Avg Loss: 0.0906
  Epoch 16, Batch 250, Group Avg Loss: 0.0879
  Epoch 16, Batch 300, Group Avg Loss: 0.0899
  Epoch 16, Batch 350, Group Avg Loss: 0.1045
--- Epoch 16 Summary ---
Average Train Loss: 0.093306
Average Val Loss:   0.205596
mIoU:               0.6464
保存最佳模型!

Epoch 17/50
  Epoch 17, Batch 50, Group Avg Loss: 0.0874
  Epoch 17, Batch 100, Group Avg Loss: 0.0868
  Epoch 17, Batch 150, Group Avg Loss: 0.0848
  Epoch 17, Batch 200, Group Avg Loss: 0.0831
  Epoch 17, Batch 250, Group Avg Loss: 0.0812
  Epoch 17, Batch 300, Group Avg Loss: 0.0841
  Epoch 17, Batch 350, Group Avg Loss: 0.0809
--- Epoch 17 Summary ---
Average Train Loss: 0.084173
Average Val Loss:   0.199987
mIoU:               0.6549
保存最佳模型!

Epoch 18/50
  Epoch 18, Batch 50, Group Avg Loss: 0.0782
  Epoch 18, Batch 100, Group Avg Loss: 0.0770
  Epoch 18, Batch 150, Group Avg Loss: 0.0758
  Epoch 18, Batch 200, Group Avg Loss: 0.0774
  Epoch 18, Batch 250, Group Avg Loss: 0.0752
  Epoch 18, Batch 300, Group Avg Loss: 0.0759
  Epoch 18, Batch 350, Group Avg Loss: 0.0752
--- Epoch 18 Summary ---
Average Train Loss: 0.076544
Average Val Loss:   0.199959
mIoU:               0.6629
保存最佳模型!

Epoch 19/50
  Epoch 19, Batch 50, Group Avg Loss: 0.0745
  Epoch 19, Batch 100, Group Avg Loss: 0.0746
  Epoch 19, Batch 150, Group Avg Loss: 0.0705
  Epoch 19, Batch 200, Group Avg Loss: 0.0722
  Epoch 19, Batch 250, Group Avg Loss: 0.0736
  Epoch 19, Batch 300, Group Avg Loss: 0.0741
  Epoch 19, Batch 350, Group Avg Loss: 0.0754
--- Epoch 19 Summary ---
Average Train Loss: 0.073780
Average Val Loss:   0.213132
mIoU:               0.6497

Epoch 20/50

[31m---------------------------------------------------------------------------[39m
[31mKeyboardInterrupt[39m                         Traceback (most recent call last)
[36mCell[39m[36m [39m[32mIn[3][39m[32m, line 509[39m
[32m    506[39m     trainer.train(num_epochs=[32m50[39m)
[32m    508[39m [38;5;28;01mif[39;00m [34m__name__[39m == [33m"[39m[33m__main__[39m[33m"[39m:
[32m--> [39m[32m509[39m     [43mmain[49m[43m([49m[43m)[49m

[36mCell[39m[36m [39m[32mIn[3][39m[32m, line 506[39m, in [36mmain[39m[34m()[39m
[32m    504[39m [38;5;66;03m# 启动训练[39;00m
[32m    505[39m trainer = SegmentationTrainer(model, train_loader, val_loader, criterion, optimizer, [38;5;28;01mNone[39;00m, device)
[32m--> [39m[32m506[39m [43mtrainer[49m[43m.[49m[43mtrain[49m[43m([49m[43mnum_epochs[49m[43m=[49m[32;43m50[39;49m[43m)[49m

[36mCell[39m[36m [39m[32mIn[3][39m[32m, line 455[39m, in [36mSegmentationTrainer.train[39m[34m(self, num_epochs)[39m
[32m    453[39m [38;5;28mprint[39m([33mf[39m[33m"[39m[38;5;130;01m\n[39;00m[33mEpoch [39m[38;5;132;01m{[39;00mepoch+[32m1[39m[38;5;132;01m}[39;00m[33m/[39m[38;5;132;01m{[39;00mnum_epochs[38;5;132;01m}[39;00m[33m"[39m)
[32m    454[39m [38;5;66;03m# train_loss = self.train_epoch()[39;00m
[32m--> [39m[32m455[39m avg_epoch_loss = [38;5;28;43mself[39;49m[43m.[49m[43mtrain_epoch[49m[43m([49m[43m)[49m
[32m    456[39m val_loss, val_miou = [38;5;28mself[39m.validate()
[32m    457[39m [38;5;66;03m# 3. 记录到列表（结构化存储）[39;00m

[36mCell[39m[36m [39m[32mIn[3][39m[32m, line 418[39m, in [36mSegmentationTrainer.train_epoch[39m[34m(self)[39m
[32m    415[39m loss.backward()
[32m    416[39m [38;5;28mself[39m.optimizer.step()
[32m--> [39m[32m418[39m loss_val = [43mloss[49m[43m.[49m[43mitem[49m[43m([49m[43m)[49m
[32m    419[39m total_loss += loss_val
[32m    420[39m running_loss += loss_val

[31mKeyboardInterrupt[39m: